# Chuẩn bị THÊM data — nhiều lát/bệnh nhân (LiTS u gan)

Cắt nhiều lát hơn (mặc định 5 dương + 5 âm / bệnh nhân ≈ **~1000 mẫu**) để fine-tune mạnh hơn.

⚠️ Lưu vào **folder MỚI** `medrega/data_multi/` — KHÔNG đè `data_liver` (249 mẫu) cũ.
Split theo bệnh nhân (làm ở notebook train) nên các lát cùng người vẫn cùng tập → **không lộ dữ liệu**.

## 0. Cài + Mount + Cấu hình

In [1]:
%pip install -q kagglehub nibabel pillow numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# === Mount Drive + cấu hình (OUTPUT folder MỚI, tách data_liver cũ) ===
from google.colab import drive
drive.mount('/content/drive')
import os

DRIVE_ROOT = "/content/drive/MyDrive/Colab Notebooks/medrega"
OUT_DIR    = os.path.join(DRIVE_ROOT, "data_multi", "data_liver")   # <-- FOLDER MỚI (không đè data cũ)
os.makedirs(os.path.join(OUT_DIR, "images"), exist_ok=True)

# Tham số cắt — tăng số lát/bệnh nhân để có nhiều data
N_POS_SLICES = 5      # số lát DƯƠNG (u lớn nhất) mỗi bệnh nhân
N_NEG_SLICES = 5      # số lát ÂM (gan không u) mỗi bệnh nhân
WL, WW       = 40, 400        # cửa sổ HU (gan/mô mềm)
COORD_SCALE  = 1000           # toạ độ box [0,1000)
print("OUTPUT (MỚI):", OUT_DIR, "| mỗi bệnh nhân:", N_POS_SLICES, "dương +", N_NEG_SLICES, "âm")

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Kaggle auth (nếu kagglehub cần): đọc KAGGLE_USERNAME/KAGGLE_KEY từ Colab secret hoặc secrets.env
import os
def _set_kaggle():
    try:
        from google.colab import userdata
        u, k = userdata.get("KAGGLE_USERNAME"), userdata.get("KAGGLE_KEY")
        if u and k:
            os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = u, k
            return "Colab secret"
    except Exception:
        pass
    env = os.path.join(DRIVE_ROOT, "secrets.env")
    if os.path.exists(env):
        for line in open(env, encoding="utf-8"):
            line = line.strip()
            if "=" in line and line.split("=", 1)[0].strip() in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
                k, v = line.split("=", 1)
                os.environ[k.strip()] = v.strip().strip(chr(34)).strip(chr(39))
        if os.environ.get("KAGGLE_KEY"):
            return "secrets.env"
    return "không có (thử tải public, nếu lỗi 403 thì thêm KAGGLE_USERNAME/KAGGLE_KEY)"
print("Kaggle auth:", _set_kaggle())

## 1. Tải LiTS

In [ ]:
# === Dùng LiTS có sẵn nếu tìm thấy, không thì tải từ Kaggle ===
import glob, os
# các chỗ hay để LiTS trên Drive (sửa nếu mày để chỗ khác)
search_dirs = [
    os.path.join(DRIVE_ROOT, "data", "LiTS"),
    os.path.join(DRIVE_ROOT, "LiTS"),
    os.path.join(DRIVE_ROOT, "data"),
    "/content/LiTS",
]
LITS_LOCAL = None
for d in search_dirs:
    if os.path.isdir(d) and glob.glob(os.path.join(d, "**", "segmentation-*.nii*"), recursive=True):
        LITS_LOCAL = d
        break

if LITS_LOCAL:
    roots = [LITS_LOCAL]
    n_seg = len(glob.glob(os.path.join(LITS_LOCAL, "**", "segmentation-*.nii*"), recursive=True))
    n_vol = len(glob.glob(os.path.join(LITS_LOCAL, "**", "volume-*.nii*"), recursive=True))
    print("Dùng LiTS CÓ SẴN:", LITS_LOCAL, "| segmentation:", n_seg, "| volume:", n_vol)
    print("(đọc .nii từ Drive sẽ chậm hơn local — kiên nhẫn ở cell cắt)")
else:
    print("Không thấy LiTS sẵn -> tải từ Kaggle về /content (nhanh)...")
    import kagglehub
    roots = [kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation"),
             kagglehub.dataset_download("andrewmvd/liver-tumor-segmentation-part-2")]

for r in roots:
    print("root:", r)

## 2. Cắt 3D → 2D (nhiều lát)

In [ ]:
# === Cắt 3D -> 2D NHIỀU lát/bệnh nhân -> data.jsonl + ảnh (lưu folder MỚI) ===
import os, re, glob, json, numpy as np, nibabel as nib
from PIL import Image

def _window(x):                                  # HU -> 0..255
    lo, hi = WL - WW / 2, WL + WW / 2
    return ((np.clip(x, lo, hi) - lo) / (hi - lo) * 255).astype(np.uint8)

def _box(mask, W, H):                            # mask -> [x1,y1,x2,y2] thang [0,1000)
    ys, xs = np.where(mask)
    return [int(xs.min() / W * COORD_SCALE), int(ys.min() / H * COORD_SCALE),
            int(xs.max() / W * COORD_SCALE), int(ys.max() / H * COORD_SCALE)]

def _rec(fn, box, label, pid):
    q = "Please detect the liver tumor in this CT image and give its bounding box."
    a = (f"<ref>liver tumor</ref><box>[[{box[0]}, {box[1]}, {box[2]}, {box[3]}]]</box>"
         if box is not None else "No liver tumor is found.")
    return {"id": f"{pid}_{label}_{os.path.basename(fn)}", "patient_id": int(pid),
            "image_path": fn, "question": q, "gt_box": box, "label": label, "modality": "ct_liver",
            "conversations": [{"from": "human", "value": "<image>\n" + q},
                              {"from": "gpt", "value": a}]}

# gộp file từ cả 2 part
seg_paths, vol_paths = {}, {}
for r in roots:
    for p in glob.glob(os.path.join(r, "**", "segmentation-*.nii*"), recursive=True):
        seg_paths[int(re.search(r"segmentation-(\d+)", p).group(1))] = p
    for p in glob.glob(os.path.join(r, "**", "volume-*.nii*"), recursive=True):
        vol_paths[int(re.search(r"volume-(\d+)", p).group(1))] = p
print("Số ca có nhãn:", len(seg_paths), "| có volume:", len(vol_paths))

rows = []
for k, pid in enumerate(sorted(seg_paths)):
    if pid not in vol_paths:
        continue
    seg = np.asarray(nib.load(seg_paths[pid]).dataobj)        # dtype gốc (nhẹ)
    vimg = nib.load(vol_paths[pid])                           # LAZY: đọc lát cần
    H, W, D = seg.shape
    tumor = (seg == 2).reshape(-1, D).sum(0)                  # px u mỗi lát
    liver = (seg == 1).reshape(-1, D).sum(0)

    def _save(z, fn):
        Image.fromarray(_window(np.asarray(vimg.dataobj[:, :, z]))).save(os.path.join(OUT_DIR, fn))

    # DƯƠNG: N_POS_SLICES lát có u LỚN NHẤT
    pos_z = [int(z) for z in np.argsort(tumor)[::-1] if tumor[z] > 0][:N_POS_SLICES]
    for z in pos_z:
        fn = f"images/liver_{pid:03d}_pos_z{z}.png"; _save(z, fn)
        rows.append(_rec(fn, _box(seg[:, :, z] == 2, W, H), "tumor", pid))

    # ÂM: N_NEG_SLICES lát gan-KHÔNG-u, rải đều
    cand = np.where((tumor == 0) & (liver > 0))[0]
    if len(cand):
        pick = cand[np.linspace(0, len(cand) - 1, min(N_NEG_SLICES, len(cand))).astype(int)]
        for z in sorted(set(int(z) for z in pick)):
            fn = f"images/liver_{pid:03d}_neg_z{z}.png"; _save(z, fn)
            rows.append(_rec(fn, None, "none", pid))

    del seg
    print(f"  [{k+1}/{len(seg_paths)}] pid {pid}: +{len(pos_z)} dương", flush=True)

with open(os.path.join(OUT_DIR, "data.jsonl"), "w", encoding="utf-8") as f:
    for r in rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print("=" * 50)
print("Tổng mẫu:", len(rows),
      "| dương:", sum(r["label"] == "tumor" for r in rows),
      "| âm:", sum(r["label"] == "none" for r in rows))
print("Số bệnh nhân:", len({r["patient_id"] for r in rows}))
print("Lưu ở (folder MỚI):", OUT_DIR)

## 3. Zip backup

In [ ]:
# === Zip để backup / upload sang notebook train ===
import shutil, os
zip_base = os.path.join(os.path.dirname(OUT_DIR), "data_liver_multi")   # medrega/data_multi/data_liver_multi.zip
shutil.make_archive(zip_base, "zip", OUT_DIR)
print("Đã zip:", zip_base + ".zip", "|", round(os.path.getsize(zip_base + ".zip")/1e6, 1), "MB")
print("=> Notebook train: đổi getdata trỏ tới data_multi/data_liver (hoặc upload zip này).")